# Адаптер следователя `inv_v1` (Colab A100 80 ГБ)

Двухфазный агент решает 86% эпизодов, когда следователь отдаёт диагноз до ~210 с, и 0%, когда не отдаёт (37% эпизодов):
модель находит нужный код к ~3-му вызову, но не решает остановиться. Адаптер учит это решение на парах
(`train/build_inv_data.py`, 681 пара по 39 задачам; 46 задач проверки исключены):
* **stop** -- нужный код уже прочитан: chosen -- диагноз FILE/LINES/CAUSE/FIX/CHECK (FILE/LINES/FIX/CHECK из эталонного патча),
  rejected -- то, что модель сделала на самом деле (ещё grep/read);
* **go** -- шаг до того, как модель увидела код: chosen -- её настоящий вызов, rejected -- преждевременный диагноз.

**Часть A** (vLLM): Gemma дописывает рассуждение и CAUSE (stop) и преждевременный диагноз (go), ~30-60 мин.
**Часть B** (на чистой среде, без vLLM): обучение `pref_lora --ref` (chosen и rejected разной длины -- без reference
длинный диагноз проигрывал бы просто числом токенов), сохранение в Drive.

В Drive (`MyDrive/gemma_adapters/`) положите `inv_pairs.jsonl` (это `data/inv_pairs.jsonl` из проекта).
Если часть A уже сделана (`inv_pairs_filled.jsonl` в Drive) -- сразу к части B.

## 0. Параметры (выполнять перед каждой частью)

In [ ]:
PAIRS = "inv_pairs.jsonl"              # в gemma_adapters: data/inv_pairs.jsonl из проекта (681 пара, заглушки)
FILLED = "inv_pairs_filled.jsonl"      # сюда часть A пишет пары с ответами Gemma
ADAPTER = "inv_v1"                     # имя адаптера следователя
EPOCHS = 1                             # 1 эпоха ~6 ч на A100 80 ГБ; 2 -- ~12 ч
MAX_LEN = 8192
FILL_WORKERS = 8

# Часть A: Gemma дописывает пары
## A1. Карта

In [ ]:
import subprocess
out = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,memory.used", "--format=csv,noheader,nounits"],
                     capture_output=True, text=True).stdout.strip()
print(out)
name, total, used = [x.strip() for x in out.split(",")]
assert int(total) >= 79000, f"нужна карта на 80 ГБ, а здесь {name} {total} МиБ: смените среду выполнения"

## A2. Код, vLLM, Drive

In [ ]:
import os
if not os.path.exists("/content/gemma4-swe-rl"):
    !git clone -q https://github.com/aletonfdg/gemma4-swe-rl.git /content/gemma4-swe-rl
%cd /content/gemma4-swe-rl
!git pull -q
!git log --oneline | head -3
!pip install -q vllm openai pyyaml requests huggingface_hub kagglehub
assert os.path.exists("train/fill_inv_data.py")

In [ ]:
import os, json, shutil
from google.colab import drive
drive.mount("/content/drive")
DRIVE = "/content/drive/MyDrive/gemma_adapters"

In [ ]:
assert os.path.exists(f"{DRIVE}/{PAIRS}"), f"положите data/inv_pairs.jsonl из проекта в {DRIVE}/{PAIRS}"
os.makedirs("data", exist_ok=True)
shutil.copy(f"{DRIVE}/{PAIRS}", "data/inv_pairs.jsonl")
kinds = {}
for l in open("data/inv_pairs.jsonl", encoding="utf-8"):
    k = json.loads(l)["kind"]; kinds[k] = kinds.get(k, 0) + 1
print("пар:", kinds)

## A3. Базовая модель

In [ ]:
import getpass
MODEL_DIR = "/content/model"
if not os.path.exists(os.path.join(MODEL_DIR, "config.json")):
    try:
        from huggingface_hub import snapshot_download
        hf_token = getpass.getpass("HF токен (Enter, если не нужен): ").strip() or None
        snapshot_download("google/gemma-4-31b-it-qat-w4a16-ct", local_dir=MODEL_DIR, token=hf_token)
    except Exception as e:
        print("HF не получилось:", type(e).__name__, str(e)[:300])
        import kagglehub
        os.environ["KAGGLE_API_TOKEN"] = getpass.getpass("KAGGLE_API_TOKEN (KGAT_...): ").strip()
        path = kagglehub.model_download("google/gemma-4/transformers/gemma-4-31b-it-qat-w4a16-ct")
        !cp -r "{path}"/. {MODEL_DIR}/
!du -sh {MODEL_DIR}

## A4. vLLM (базовая модель, без адаптеров)

In [ ]:
import subprocess, time, requests
BASE = "gemma-4-31b-it-qat-w4a16-ct"
cmd = ["vllm", "serve", MODEL_DIR, "--served-model-name", BASE,
       "--quantization", "compressed-tensors", "--dtype", "bfloat16",
       "--max-model-len", "32768", "--gpu-memory-utilization", "0.92", "--max-num-seqs", str(FILL_WORKERS),
       "--enable-auto-tool-choice", "--tool-call-parser", "gemma4", "--reasoning-parser", "gemma4",
       "--default-chat-template-kwargs", '{"enable_thinking": true}', "--port", "8000"]
proc = subprocess.Popen(cmd, stdout=open("/content/vllm.log", "w"), stderr=subprocess.STDOUT, start_new_session=True)
t0 = time.time()
while time.time() - t0 < 1500 and proc.poll() is None:
    try:
        if requests.get("http://127.0.0.1:8000/health", timeout=3).status_code == 200:
            print("vLLM готов за %.0f c" % (time.time() - t0)); break
    except requests.RequestException:
        pass
    time.sleep(5)
if proc.poll() is not None:
    print("vLLM упал:"); print(open("/content/vllm.log").read()[-3000:])

## A5. Заполнение (лог: каждые 50 пар -- сколько готово и сколько не удалось)

In [ ]:
!python -m train.fill_inv_data --data data/inv_pairs.jsonl --out data/{FILLED} --workers {FILL_WORKERS} 2>&1 | tee /content/fill.log
shutil.copy(f"data/{FILLED}", f"{DRIVE}/{FILLED}")
print("в Drive:", f"{DRIVE}/{FILLED}")

In [ ]:
# три примера: глазами проверить, что диагноз осмысленный, а рассуждение не упоминает эталон
import random
rows = [json.loads(l) for l in open(f"data/{FILLED}", encoding="utf-8")]
random.seed(0)
for r in random.sample([r for r in rows if r["kind"] == "stop"], 2) + random.sample([r for r in rows if r["kind"] == "go"], 1):
    last = r["chosen"][-1] if r["kind"] == "stop" else r["rejected"][-1]
    print("=" * 20, r["kind"], r["task"]); print("THOUGHT:", last.get("thought", "")[:600]); print(last["content"][:1500])

# Часть B: обучение
**Сначала: Среда выполнения -> Отключиться и удалить среду выполнения**, затем снова A100 80 ГБ и ячейки: 0, B1...
(vLLM обновил torch; обучение идёт на родном torch Colab и без vLLM в памяти).
## B1. Карта, код, Drive

In [ ]:
import subprocess
out = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,memory.used", "--format=csv,noheader,nounits"],
                     capture_output=True, text=True).stdout.strip()
print(out)
name, total, used = [x.strip() for x in out.split(",")]
assert int(total) >= 79000, f"нужна карта на 80 ГБ, а здесь {name} {total} МиБ: смените среду выполнения"

In [ ]:
import os
if not os.path.exists("/content/gemma4-swe-rl"):
    !git clone -q https://github.com/aletonfdg/gemma4-swe-rl.git /content/gemma4-swe-rl
%cd /content/gemma4-swe-rl
!git pull -q
!git log --oneline | head -3
# torch и transformers в Colab уже есть; не обновляем их, чтобы не сломать совместимость
!pip install -q compressed-tensors bitsandbytes accelerate safetensors huggingface_hub kagglehub pyyaml

In [ ]:
import os, json, shutil
from google.colab import drive
drive.mount("/content/drive")
DRIVE = "/content/drive/MyDrive/gemma_adapters"

In [ ]:
assert os.path.exists(f"{DRIVE}/{FILLED}"), "нет пар с ответами: сначала часть A"
os.makedirs("data", exist_ok=True)
shutil.copy(f"{DRIVE}/{FILLED}", f"data/{FILLED}")
out = subprocess.run(["nvidia-smi", "--query-gpu=memory.used", "--format=csv,noheader,nounits"], capture_output=True, text=True).stdout
assert int(out.strip()) < 2000, "карта занята (vLLM?): перезапустите среду -- обучение идёт без vLLM"

## B2. Базовая модель

In [ ]:
import getpass
MODEL_DIR = "/content/model"
if not os.path.exists(os.path.join(MODEL_DIR, "config.json")):
    try:
        from huggingface_hub import snapshot_download
        hf_token = getpass.getpass("HF токен (Enter, если не нужен): ").strip() or None
        snapshot_download("google/gemma-4-31b-it-qat-w4a16-ct", local_dir=MODEL_DIR, token=hf_token)
    except Exception as e:
        print("HF не получилось:", type(e).__name__, str(e)[:300])
        import kagglehub
        os.environ["KAGGLE_API_TOKEN"] = getpass.getpass("KAGGLE_API_TOKEN (KGAT_...): ").strip()
        path = kagglehub.model_download("google/gemma-4/transformers/gemma-4-31b-it-qat-w4a16-ct")
        !cp -r "{path}"/. {MODEL_DIR}/
!du -sh {MODEL_DIR}

## B3. Обучение (~6 ч на 1 эпоху)
В логе: `пар: N` (сколько влезло в MAX_LEN), `eval до/в конце` по видам `stop` и `go` -- доля пар chosen>rejected
(с `--ref` -- относительно базовой модели; до обучения около 0.5). Хорошо: обе доли в конце >= 0.9.
Если `go` сильно ниже `stop` -- адаптер учится отвечать слишком рано: напишите мне.

In [ ]:
LOG = f"/content/{ADAPTER}.log"
!python -m train.pref_lora --data data/{FILLED} --name {ADAPTER} --decompress --ref --epochs {EPOCHS} --max-len {MAX_LEN} 2>&1 | tee {LOG}

## B4. Сохранить адаптер и лог в Drive

In [ ]:
assert os.path.exists(f"adapters/{ADAPTER}/adapter_model.safetensors"), "адаптер не сохранился, смотрите лог"
!cp -r adapters/{ADAPTER} {DRIVE}/ && cp {LOG} {DRIVE}/
!ls -la {DRIVE}/{ADAPTER}
!grep -E "пар:|eval" {LOG}